In [6]:
# -*- coding: utf-8 -*-
import math
import random
import csv
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import requests
from io import StringIO
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth import default
import tempfile

# URL do Google Sheets para o CSV exportado
url = "https://docs.google.com/spreadsheets/d/1FUx1nRvhRRKhwYDwxUXMutXD-17lQ4I-pSZ27ERkC14/export?format=csv&gid=1161976353"

# Autenticação no Google Drive
def authenticate_google_drive():
    print("Autenticando no Google Drive...")
    auth.authenticate_user()
    creds, _ = default()
    drive_service = build('drive', 'v3', credentials=creds)
    return drive_service

# Função para carregar o dataset diretamente do URL
def load_dataset_from_url(url):
    response = requests.get(url)
    if response.status_code == 200:
        data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(data), sep=",")
        df = df.replace({',': '.'}, regex=True)
        cols_to_convert = df.columns[2:]
        df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce')
        return df
    else:
        print("Erro ao acessar a URL. Código de status:", response.status_code)
        return None

# Função para calcular o sumário estatístico e gerar MANCOVA
def statistical_summary_and_mancova(df):
    print('Sumário Estatístico Detalhado\n')

    males = df[df['Sexo'] == 'Masculino']
    females = df[df['Sexo'] == 'Femenino']

    variables = ['Sociosexualidade_Geral', 'Atitude', 'Desejo', 'Promocao']
    summary = ""

    # Sumário estatístico para cada variável
    for var in variables:
        summary += f'\n{var}:\n'
        summary += f'Média (Homens): {males[var].mean():.4f}\n'
        summary += f'Desvio Padrão (Homens): {males[var].std():.4f}\n'
        summary += f'Média (Mulheres): {females[var].mean():.4f}\n'
        summary += f'Desvio Padrão (Mulheres): {females[var].std():.4f}\n'

        t_stat, p_value = stats.ttest_ind(males[var].dropna(), females[var].dropna())
        summary += f'\nTeste t ({var}): t={t_stat:.4f}, p={p_value:.4f}\n'

    # Correlação entre Sociosexualidade_Geral e Desejo
    correlation, p_value = stats.pearsonr(df['Sociosexualidade_Geral'].dropna(), df['Desejo'].dropna())
    summary += f'\nCorrelação entre Sociosexualidade_Geral e Desejo: r={correlation:.4f}, p={p_value:.4f}\n'

    # MANCOVA
    manova = MANOVA.from_formula('Sociosexualidade_Geral + Atitude + Desejo + Promocao ~ Sexo', data=df)
    mancova_results = manova.mv_test()
    summary += "\nMANCOVA Results:\n"
    summary += str(mancova_results)

    # Post-Hoc Tukey HSD
    for var in variables:
        tukey = pairwise_tukeyhsd(endog=df[var].dropna(), groups=df['Sexo'].dropna(), alpha=0.05)
        summary += f"\nPost-Hoc Results (Tukey HSD) for {var}:\n"
        summary += tukey.summary().as_text() + "\n"

    return summary

# Função para salvar o arquivo CSV no Google Drive
def save_to_drive(drive_service, filename, content):
    print("\nSalvando arquivo no Google Drive...")

    # Cria um arquivo temporário no disco para salvar o CSV
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv') as temp_file:
        temp_file_name = temp_file.name
        content.to_csv(temp_file, index=False)

    # Define o nome do arquivo e o tipo MIME
    file_metadata = {'name': filename, 'mimeType': 'text/csv'}

    # Cria o MediaFileUpload a partir do arquivo temporário
    media = MediaFileUpload(temp_file_name, mimetype='text/csv')

    # Envia o arquivo para o Google Drive
    file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print(f"Arquivo '{filename}' salvo no Google Drive com ID: {file.get('id')}\n")

# Função para gerar o sumário descritivo para salvar
def generate_summary_file(summary):
    print("\nGerando sumário descritivo...")
    df_summary = pd.DataFrame([summary], columns=["Summary"])
    return df_summary

# Função principal
def main():
    # Autentica no Google Drive
    drive_service = authenticate_google_drive()

    # Carrega o dataset do URL
    df = load_dataset_from_url(url)

    if df is not None:
        # Realiza o sumário estatístico e MANCOVA
        summary = statistical_summary_and_mancova(df)

        # Gera o arquivo de sumário descritivo
        summary_df = generate_summary_file(summary)

        # Salva o arquivo no Google Drive
        save_to_drive(drive_service, "sumario-Sociosexualidade_Geral-Atitude-Desejo-Promocao.csv", summary_df)

if __name__ == "__main__":
    main()

Autenticando no Google Drive...
Sumário Estatístico Detalhado


Gerando sumário descritivo...

Salvando arquivo no Google Drive...
Arquivo 'sumario-Sociosexualidade_Geral-Atitude-Desejo-Promocao.csv' salvo no Google Drive com ID: 1zkJ8VqHEAGTucepx6hwTQTAX_rrgh75C

